In [37]:
import pandas as pd
from collections import Counter

df = pd.read_csv("../data/jobs_with_skills.csv")
df['extracted_skills'] = df['extracted_skills'].apply(eval)  # string ko list me convert karo
df.shape

(8713, 17)

In [38]:
def get_top_skills_for_role(role_keyword, top_n=10):
    # Us keyword wale titles filter karo (case-insensitive match)
    role_jobs = df[df['title'].str.contains(role_keyword, case=False, na=False)]
    
    if len(role_jobs) == 0:
        return None, 0
    
    all_skills = role_jobs['extracted_skills'].sum()
    skill_counts = Counter(all_skills)
    top_skills = [skill for skill, count in skill_counts.most_common(top_n)]
    
    return top_skills, len(role_jobs)

# Test karo
top_skills, num_jobs = get_top_skills_for_role("Data Analyst")
print(f"Found {num_jobs} jobs")
print("Top skills:", top_skills)

Found 28 jobs
Top skills: ['excel', 'sql', 'communication', 'leadership', 'data analysis', 'tableau', 'power bi', 'project management', 'marketing', 'python']


In [39]:
def skill_gap_analysis(target_role, my_skills, top_n=10):
    # User ki skills ko lowercase me convert karo
    my_skills = [s.strip().lower() for s in my_skills]

    required_skills, num_jobs = get_top_skills_for_role(target_role, top_n)

    if required_skills is None:
        print(f"No jobs found for role: {target_role}")
        return

    matched = [s for s in required_skills if s in my_skills]
    missing = [s for s in required_skills if s not in my_skills]
    match_pct = (len(matched) / len(required_skills)) * 100

    print(f"Target Role: {target_role} (based on {num_jobs} job postings)")
    print(f"Top skills required: {required_skills}")
    print()
    print(f"You have: {matched}")
    print(f"You are missing: {missing}")
    print(f"Skill Match: {match_pct:.0f}%")

In [40]:
skill_gap_analysis("Data Analyst", ["python", "excel", "communication"])

Target Role: Data Analyst (based on 28 job postings)
Top skills required: ['excel', 'sql', 'communication', 'leadership', 'data analysis', 'tableau', 'power bi', 'project management', 'marketing', 'python']

You have: ['excel', 'communication', 'python']
You are missing: ['sql', 'leadership', 'data analysis', 'tableau', 'power bi', 'project management', 'marketing']
Skill Match: 30%


In [41]:
skill_gap_analysis("Nurse", ["communication", "nursing"])

Target Role: Nurse (based on 232 job postings)
Top skills required: ['nursing', 'communication', 'leadership', 'customer service', 'excel', 'marketing', 'html']

You have: ['nursing', 'communication']
You are missing: ['leadership', 'customer service', 'excel', 'marketing', 'html']
Skill Match: 29%


In [42]:
import re
nurse = df[df['title'].str.contains('Nurse', case=False, na=False)]
html_jobs = nurse[nurse['extracted_skills'].apply(lambda s: 'html' in s)]
print(len(html_jobs), "nurse jobs me 'html' mila")
for d in html_jobs['description'].head(3):
    m = re.search(r'.{40}html.{40}', d, flags=re.I | re.S)
    print(m.group(0) if m else None)
    print('---')

1 nurse jobs me 'html' mila
t: https://statejobs.nebraska.gov/index.html#benefits

Equal Opportunity Statement


---


In [43]:
skill_gap_analysis("Data Analyst", ["python", "excel", "communication"])

Target Role: Data Analyst (based on 28 job postings)
Top skills required: ['excel', 'sql', 'communication', 'leadership', 'data analysis', 'tableau', 'power bi', 'project management', 'marketing', 'python']

You have: ['excel', 'communication', 'python']
You are missing: ['sql', 'leadership', 'data analysis', 'tableau', 'power bi', 'project management', 'marketing']
Skill Match: 30%


In [44]:
def extract_skills(text):
    text = str(text).lower()
    found = []
    for skill in skills_list:
        pattern = r'(?<![a-z0-9.])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text):
            found.append(skill)
    return found